# I**nceptionResNetV2**

In [12]:
# =================== IMPORTS ===================
import os, shutil, random
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.applications.inception_resnet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# =================== CONFIG ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 60
K_FOLDS = 5
INITIAL_LR = 5e-5
LABEL_SMOOTHING = 0.1
SEED = 42

original_dataset_dir = '/kaggle/input/colon_augmented_png_v9'
output_base_dir = '/kaggle/working/inception_kfold'
train_val_dir = os.path.join(output_base_dir, 'train_val')
test_dir = os.path.join(output_base_dir, 'test')

# =================== CLEAN FOLDER ===================
if os.path.exists(output_base_dir):
    shutil.rmtree(output_base_dir)
os.makedirs(train_val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# =================== SPLIT TEST SET ===================
class_names = sorted(os.listdir(original_dataset_dir))
image_paths, labels = [], []

for label_idx, class_name in enumerate(class_names):
    img_dir = os.path.join(original_dataset_dir, class_name)
    imgs = sorted(os.listdir(img_dir))
    train_val_imgs, test_imgs = train_test_split(imgs, test_size=0.15, random_state=SEED)

    os.makedirs(os.path.join(train_val_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    for img in train_val_imgs:
        src = os.path.join(img_dir, img)
        dst = os.path.join(train_val_dir, class_name, img)
        shutil.copy(src, dst)
        image_paths.append(dst)
        labels.append(label_idx)

    for img in test_imgs:
        shutil.copy(os.path.join(img_dir, img), os.path.join(test_dir, class_name, img))

image_paths, labels = np.array(image_paths), np.array(labels)

# =================== TEST GENERATOR ===================
test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

# =================== MODEL FUNCTION ===================
def build_model():
    base_model = InceptionResNetV2(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    base_model.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = layers.Dropout(0.6)(x)
    outputs = layers.Dense(len(class_names), activation='softmax')(x)

    return models.Model(inputs, outputs)

# =================== CROSS-VALIDATION ===================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels)):
    print(f"\n\U0001F4C1 Fold {fold+1}/{K_FOLDS}")

    fold_dir = os.path.join(output_base_dir, f'fold_{fold+1}')
    train_dir = os.path.join(fold_dir, 'train')
    val_dir = os.path.join(fold_dir, 'val')
    for base in [train_dir, val_dir]:
        for class_name in class_names:
            os.makedirs(os.path.join(base, class_name), exist_ok=True)

    for split_name, split_idx in [('train', train_idx), ('val', val_idx)]:
        for i in split_idx:
            class_name = class_names[labels[i]]
            src = image_paths[i]
            dst = os.path.join(fold_dir, split_name, class_name, os.path.basename(src))
            shutil.copy(src, dst)

    train_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
        train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
    )
    val_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
        val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
    )

    model = build_model()
    loss_fn = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
    model.compile(optimizer=Adam(learning_rate=INITIAL_LR), loss=loss_fn, metrics=['accuracy'])

    checkpoint_cb = callbacks.ModelCheckpoint(
        os.path.join(fold_dir, 'inceptionresnetv2.keras'), monitor='val_loss', save_best_only=True, verbose=1
    )
    earlystop_cb = callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

    model.fit(
        train_gen, validation_data=val_gen, epochs=EPOCHS,
        callbacks=[checkpoint_cb, earlystop_cb], verbose=1
    )

    model.load_weights(os.path.join(fold_dir, 'inceptionresnetv2.keras'))
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    print(f"✅ Fold {fold+1} Test Accuracy: {test_acc:.4f}")
    fold_accuracies.append(test_acc)

# =================== FINAL RESULT ===================
print("\n\U0001F4CA K-Fold Test Accuracies:")
for i, acc in enumerate(fold_accuracies):
    print(f"  Fold {i+1}: {acc:.4f}")
print(f"\n✅ Average Test Accuracy: {np.mean(fold_accuracies):.4f}")


Found 900 images belonging to 4 classes.

📁 Fold 1/5
Found 4080 images belonging to 4 classes.
Found 1020 images belonging to 4 classes.
Epoch 1/60


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step - accuracy: 0.3749 - loss: 1.8658
Epoch 1: val_loss improved from inf to 0.87891, saving model to /kaggle/working/inception_kfold/fold_1/inceptionresnetv2.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 86s 419ms/step - accuracy: 0.3757 - loss: 1.8627 - val_accuracy: 0.7735 - val_loss: 0.8789
Epoch 2/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.6575 - loss: 0.9934
Epoch 2: val_loss improved from 0.87891 to 0.76989, saving model to /kaggle/working/inception_kfold/fold_1/inceptionresnetv2.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 29s 223ms/step - accuracy: 0.6577 - loss: 0.9930 - val_accuracy: 0.8157 - val_loss: 0.7699
Epoch 3/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.7422 - loss: 0.8605
Epoch 3: val_loss improved from 0.76989 to 0.69897, saving model to /kaggle/working/inception_kfold/fold_1/inceptionresnetv2.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 28s 219ms/step - accuracy: 0.7424 - loss: 0.8604 - val_accuracy: 0.8598 - val_loss: 0.699

# Xception

In [13]:
import os, shutil, random
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# ========== CONFIG ==========
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 60
K_FOLDS = 5
INITIAL_LR = 1e-4
LABEL_SMOOTHING = 0.1
SEED = 42

original_dataset_dir = '/kaggle/input/colon_augmented_png_v9'
output_base_dir = '/kaggle/working/xception_kfold'
train_val_dir = os.path.join(output_base_dir, 'train_val')
test_dir = os.path.join(output_base_dir, 'test')

# ========== CLEAN OUTPUT ==========
if os.path.exists(output_base_dir):
    shutil.rmtree(output_base_dir)
os.makedirs(train_val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# ========== SPLIT TEST SET ==========
class_names = sorted(os.listdir(original_dataset_dir))
image_paths, labels = [], []

for label_idx, class_name in enumerate(class_names):
    img_dir = os.path.join(original_dataset_dir, class_name)
    imgs = sorted(os.listdir(img_dir))
    train_val_imgs, test_imgs = train_test_split(imgs, test_size=0.15, random_state=SEED)

    os.makedirs(os.path.join(train_val_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    for img in train_val_imgs:
        src = os.path.join(img_dir, img)
        dst = os.path.join(train_val_dir, class_name, img)
        shutil.copy(src, dst)
        image_paths.append(dst)
        labels.append(label_idx)

    for img in test_imgs:
        shutil.copy(os.path.join(img_dir, img), os.path.join(test_dir, class_name, img))

image_paths = np.array(image_paths)
labels = np.array(labels)

# ========== TEST GENERATOR ==========
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

# ========== MODEL FUNCTION ==========
def build_xception_model(num_classes):
    base_model = Xception(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    base_model.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', kernel_regularizer=l2(1e-4))(x)

    return models.Model(inputs, outputs)

# ========== CROSS-VALIDATION ==========
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels)):
    print(f"\n📁 Fold {fold+1}/{K_FOLDS}")

    fold_dir = os.path.join(output_base_dir, f'fold_{fold+1}')
    train_dir = os.path.join(fold_dir, 'train')
    val_dir = os.path.join(fold_dir, 'val')
    for base in [train_dir, val_dir]:
        for class_name in class_names:
            os.makedirs(os.path.join(base, class_name), exist_ok=True)

    for split_name, split_idx in [('train', train_idx), ('val', val_idx)]:
        for i in split_idx:
            class_name = class_names[labels[i]]
            src = image_paths[i]
            dst = os.path.join(fold_dir, split_name, class_name, os.path.basename(src))
            shutil.copy(src, dst)

    train_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
        train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
    )
    val_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
        val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
    )

    model = build_xception_model(num_classes=len(class_names))
    model.compile(
        optimizer=Adam(learning_rate=INITIAL_LR),
        loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=['accuracy']
    )

    checkpoint_cb = callbacks.ModelCheckpoint(
        os.path.join(fold_dir, 'xception.keras'),
        monitor='val_loss', save_best_only=True, verbose=1
    )
    earlystop_cb = callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

    model.fit(
        train_gen, validation_data=val_gen, epochs=EPOCHS,
        callbacks=[checkpoint_cb, earlystop_cb], verbose=1
    )

    model.load_weights(os.path.join(fold_dir, 'xception.keras'))
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    print(f"✅ Fold {fold+1} Test Accuracy: {test_acc:.4f}")
    fold_accuracies.append(test_acc)

# ========== FINAL RESULTS ==========
print("\n📊 K-Fold Test Accuracies:")
for i, acc in enumerate(fold_accuracies):
    print(f"  Fold {i+1}: {acc:.4f}")
print(f"\n✅ Average Test Accuracy: {np.mean(fold_accuracies):.4f}")


Found 900 images belonging to 4 classes.

📁 Fold 1/5
Found 4080 images belonging to 4 classes.
Found 1020 images belonging to 4 classes.
83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/60


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step - accuracy: 0.4929 - loss: 1.2810
Epoch 1: val_loss improved from inf to 0.69477, saving model to /kaggle/working/xception_kfold/fold_1/xception.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 64s 350ms/step - accuracy: 0.4941 - loss: 1.2791 - val_accuracy: 0.8775 - val_loss: 0.6948
Epoch 2/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.8221 - loss: 0.7774
Epoch 2: val_loss improved from 0.69477 to 0.61979, saving model to /kaggle/working/xception_kfold/fold_1/xception.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 26s 203ms/step - accuracy: 0.8222 - loss: 0.7772 - val_accuracy: 0.9078 - val_loss: 0.6198
Epoch 3/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.8684 - loss: 0.6934
Epoch 3: val_loss improved from 0.61979 to 0.59172, saving model to /kaggle/working/xception_kfold/fold_1/xception.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 27s 208ms/step - accuracy: 0.8684 - loss: 0.6933 - val_accuracy: 0.9216 - val_loss: 0.5917
Epoch 4/60
128/128 ━━━━━━━━━

# ConvNextTiny

In [1]:
import os
import shutil
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import ConvNeXtTiny
from tensorflow.keras.applications.convnext import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam

# ========== CONFIG ==========
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 60
K_FOLDS = 5
INITIAL_LR = 5e-5
LABEL_SMOOTHING = 0.1
SEED = 42

original_dataset_dir = '/kaggle/input/colon_augmented_png_v9'
output_base_dir = '/kaggle/working/convnext_kfold'
train_val_dir = os.path.join(output_base_dir, 'train_val')
test_dir = os.path.join(output_base_dir, 'test')

# ========== CLEAN OUTPUT ==========
if os.path.exists(output_base_dir):
    shutil.rmtree(output_base_dir)
os.makedirs(train_val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# ========== SPLIT TEST SET ==========
class_names = sorted(os.listdir(original_dataset_dir))
image_paths, labels = [], []

for label_idx, class_name in enumerate(class_names):
    img_dir = os.path.join(original_dataset_dir, class_name)
    imgs = sorted(os.listdir(img_dir))
    train_val_imgs, test_imgs = train_test_split(imgs, test_size=0.15, random_state=SEED)

    os.makedirs(os.path.join(train_val_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    for img in train_val_imgs:
        src = os.path.join(img_dir, img)
        dst = os.path.join(train_val_dir, class_name, img)
        shutil.copy(src, dst)
        image_paths.append(dst)
        labels.append(label_idx)

    for img in test_imgs:
        shutil.copy(os.path.join(img_dir, img), os.path.join(test_dir, class_name, img))

image_paths = np.array(image_paths)
labels = np.array(labels)

# ========== TEST GENERATOR ==========
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
    test_dir, 
    target_size=IMG_SIZE, 
    batch_size=BATCH_SIZE,
    class_mode='categorical', 
    shuffle=False
)

# ========== MODEL FUNCTION ==========
def build_convnexttiny_model(num_classes):
    base_model = ConvNeXtTiny(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    base_model.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(2, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)

# ========== CROSS-VALIDATION ==========
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels)):
    print(f"\n📁 Fold {fold+1}/{K_FOLDS}")

    # Create fold directories
    fold_dir = os.path.join(output_base_dir, f'fold_{fold+1}')
    train_dir = os.path.join(fold_dir, 'train')
    val_dir = os.path.join(fold_dir, 'val')
    
    for base in [train_dir, val_dir]:
        for class_name in class_names:
            os.makedirs(os.path.join(base, class_name), exist_ok=True)

    # Copy images to fold directories
    for i in train_idx:
        class_name = class_names[labels[i]]
        src = image_paths[i]
        dst = os.path.join(train_dir, class_name, os.path.basename(src))
        shutil.copy(src, dst)
        
    for i in val_idx:
        class_name = class_names[labels[i]]
        src = image_paths[i]
        dst = os.path.join(val_dir, class_name, os.path.basename(src))
        shutil.copy(src, dst)

    # Create data generators
    train_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
        train_dir, 
        target_size=IMG_SIZE, 
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )
    
    val_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
        val_dir, 
        target_size=IMG_SIZE, 
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )

    # Build and compile model
    model = build_convnexttiny_model(num_classes=len(class_names))
    model.compile(
        optimizer=Adam(learning_rate=INITIAL_LR),
        loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=['accuracy']
    )

    # Callbacks
    checkpoint_cb = callbacks.ModelCheckpoint(
        os.path.join(fold_dir, 'convnexttiny.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
    earlystop_cb = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True
    )
    reduce_lr_cb = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1,
        min_lr=1e-7
    )

    # Training
    print(f"🚀 Training Fold {fold+1}...")
    model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        callbacks=[checkpoint_cb, earlystop_cb, reduce_lr_cb],
        verbose=1
    )

    # Evaluation
    model.load_weights(os.path.join(fold_dir, 'convnexttiny.keras'))
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    print(f"✅ Fold {fold+1} Test Accuracy: {test_acc:.4f}")
    fold_accuracies.append(test_acc)

# ========== FINAL RESULTS ==========
print("\n📊 K-Fold Test Accuracies:")
for i, acc in enumerate(fold_accuracies):
    print(f"  Fold {i+1}: {acc:.4f}")
print(f"\n✅ Average Test Accuracy: {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")

2025-07-20 10:50:26.217896: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753008626.579314      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753008626.681818      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Found 900 images belonging to 4 classes.

📁 Fold 1/5
Found 4080 images belonging to 4 classes.
Found 1020 images belonging to 4 classes.


I0000 00:00:1753008701.335123      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1753008701.335811      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
🚀 Training Fold 1...


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


...

128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - accuracy: 0.2311 - loss: 1.3865

E0000 00:00:1753008752.603010     117 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1753008752.738936     117 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.



Epoch 1: val_loss improved from inf to 1.34131, saving model to /kaggle/working/convnext_kfold/fold_1/convnexttiny.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 51s 248ms/step - accuracy: 0.2314 - loss: 1.3864 - val_accuracy: 0.3882 - val_loss: 1.3413 - learning_rate: 5.0000e-05
Epoch 2/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.3934 - loss: 1.3328
Epoch 2: val_loss improved from 1.34131 to 1.30103, saving model to /kaggle/working/convnext_kfold/fold_1/convnexttiny.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 21s 161ms/step - accuracy: 0.3934 - loss: 1.3327 - val_accuracy: 0.4118 - val_loss: 1.3010 - learning_rate: 5.0000e-05
Epoch 3/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.4064 - loss: 1.2988
Epoch 3: val_loss improved from 1.30103 to 1.27380, saving model to /kaggle/working/convnext_kfold/fold_1/convnexttiny.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 22s 168ms/step - accuracy: 0.4065 - loss: 1.2988 - val_accuracy: 0.4412 - val_loss: 1.2738 - learning_rate: 5.0000e-05
Epoch 4/

E0000 00:00:1753010083.812038     117 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1753010083.946577     117 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


✅ Fold 1 Test Accuracy: 0.7511

📁 Fold 2/5
Found 4080 images belonging to 4 classes.
Found 1020 images belonging to 4 classes.
🚀 Training Fold 2...
Epoch 1/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - accuracy: 0.2551 - loss: 1.4473
Epoch 1: val_loss improved from inf to 1.34583, saving model to /kaggle/working/convnext_kfold/fold_2/convnexttiny.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 47s 246ms/step - accuracy: 0.2553 - loss: 1.4470 - val_accuracy: 0.3461 - val_loss: 1.3458 - learning_rate: 5.0000e-05
Epoch 2/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - accuracy: 0.3819 - loss: 1.3086
Epoch 2: val_loss improved from 1.34583 to 1.24233, saving model to /kaggle/working/convnext_kfold/fold_2/convnexttiny.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 23s 182ms/step - accuracy: 0.3820 - loss: 1.3085 - val_accuracy: 0.4500 - val_loss: 1.2423 - learning_rate: 5.0000e-05
Epoch 3/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.4697 - loss: 1.2103
Epoch 3: val_loss improved from 1.24233 to 1.1

# ResNet101

In [1]:
# ==================== IMPORTS ====================
import os
import numpy as np
import random
import shutil
import time
import tensorflow as tf
from tqdm import tqdm
from tensorflow.keras import layers, Model, callbacks, backend as K
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.applications.resnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import StratifiedKFold, train_test_split

# ==================== CONFIG ====================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 60
INITIAL_LR = 5e-5
LABEL_SMOOTHING = 0.1
K_FOLDS = 5
SEED = 42

# =================== PATHS ===================
dataset_dir = '/kaggle/input/colon_augmented_png_v9'
output_base_dir = '/kaggle/working/resnet101_kfold'
train_val_dir = os.path.join(output_base_dir, 'train_val')
test_dir = os.path.join(output_base_dir, 'test')

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(output_base_dir):
    shutil.rmtree(output_base_dir)

# =================== SPLIT DATA INTO TRAIN_VAL AND TEST ===================
os.makedirs(train_val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

class_names = sorted(os.listdir(dataset_dir))
TEST_RATIO = 0.15

print("Splitting dataset into train_val and test sets...")
for class_name in class_names:
    class_input_dir = os.path.join(dataset_dir, class_name)
    images = sorted(os.listdir(class_input_dir))

    train_val_imgs, test_imgs = train_test_split(images, test_size=TEST_RATIO, random_state=SEED, shuffle=True)

    os.makedirs(os.path.join(train_val_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    for img in train_val_imgs:
        shutil.copy(os.path.join(class_input_dir, img), os.path.join(train_val_dir, class_name, img))
    for img in test_imgs:
        shutil.copy(os.path.join(class_input_dir, img), os.path.join(test_dir, class_name, img))

print("✅ Dataset split done.")
print(f"Train_Val directory: {train_val_dir}")
print(f"Test directory: {test_dir}")

# =================== COLLECT TRAIN_VAL IMAGE PATHS AND LABELS ===================
image_paths = []
labels = []
for idx, class_name in enumerate(class_names):
    class_dir = os.path.join(train_val_dir, class_name)
    imgs = os.listdir(class_dir)
    for img in imgs:
        image_paths.append(os.path.join(class_dir, img))
        labels.append(idx)

image_paths = np.array(image_paths)
labels = np.array(labels)

# =================== MODEL FUNCTION ===================
def build_model():
    base_model = ResNet101(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    base_model.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(len(class_names), activation='softmax')(x)

    return Model(inputs, outputs)

# =================== DATA GENERATORS ===================
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== STRATIFIED K-FOLD TRAINING ===================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels)):
    print(f"\n📁 Fold {fold + 1}/{K_FOLDS}")

    fold_dir = os.path.join(output_base_dir, f'fold_{fold}')
    train_dir = os.path.join(fold_dir, 'train')
    val_dir = os.path.join(fold_dir, 'val')

    for dir_path in [train_dir, val_dir]:
        for class_name in class_names:
            os.makedirs(os.path.join(dir_path, class_name), exist_ok=True)

    # Copy training data
    for i in train_idx:
        src = image_paths[i]
        class_name = class_names[labels[i]]
        dst = os.path.join(train_dir, class_name, os.path.basename(src))
        shutil.copy(src, dst)

    # Copy validation data
    for i in val_idx:
        src = image_paths[i]
        class_name = class_names[labels[i]]
        dst = os.path.join(val_dir, class_name, os.path.basename(src))
        shutil.copy(src, dst)

    # Generators
    train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=True
    )
    val_generator = val_test_datagen.flow_from_directory(
        val_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    model = build_model()
    optimizer = Adam(learning_rate=INITIAL_LR)
    loss_fn = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

    checkpoint_path = os.path.join(fold_dir, 'resnet101.keras')
    checkpoint_cb = callbacks.ModelCheckpoint(
        checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1
    )
    earlystop_cb = callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1
    )

    model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        callbacks=[checkpoint_cb, earlystop_cb],
        verbose=1
    )

    model.load_weights(checkpoint_path)
    test_loss, test_acc = model.evaluate(test_generator, verbose=1)
    print(f"✅ Fold {fold + 1} Test Accuracy: {test_acc:.4f}")
    fold_accuracies.append(test_acc)

    del model
    K.clear_session()

# =================== RESULTS ===================
print("\n📊 Cross-Validation Results:")
for i, acc in enumerate(fold_accuracies):
    print(f"  Fold {i+1}: {acc:.4f}")

print(f"\n✅ Average Test Accuracy: {np.mean(fold_accuracies):.4f}")


2025-07-30 15:40:38.327779: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753890038.499812      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753890038.556775      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Splitting dataset into train_val and test sets...
✅ Dataset split done.
Train_Val directory: /kaggle/working/resnet101_kfold/train_val
Test directory: /kaggle/working/resnet101_kfold/test
Found 900 images belonging to 4 classes.

📁 Fold 1/5
Found 4080 images belonging to 4 classes.
Found 1020 images belonging to 4 classes.


I0000 00:00:1753890081.159150      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1753890081.159912      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/60


I0000 00:00:1753890104.542656     116 service.cc:148] XLA service 0x7e819c002210 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753890104.543423     116 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1753890104.543447     116 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1753890107.245642     116 cuda_dnn.cc:529] Loaded cuDNN version 90300


  1/128 ━━━━━━━━━━━━━━━━━━━━ 55:29 26s/step - accuracy: 0.2812 - loss: 2.1387

I0000 00:00:1753890112.407603     116 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - accuracy: 0.3129 - loss: 2.0548
Epoch 1: val_accuracy improved from -inf to 0.62157, saving model to /kaggle/working/resnet101_kfold/fold_0/resnet101.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 69s 334ms/step - accuracy: 0.3132 - loss: 2.0535 - val_accuracy: 0.6216 - val_loss: 1.0309
Epoch 2/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.4794 - loss: 1.4952
Epoch 2: val_accuracy improved from 0.62157 to 0.76275, saving model to /kaggle/working/resnet101_kfold/fold_0/resnet101.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 26s 199ms/step - accuracy: 0.4798 - loss: 1.4945 - val_accuracy: 0.7627 - val_loss: 0.8089
Epoch 3/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.6045 - loss: 1.2118
Epoch 3: val_accuracy improved from 0.76275 to 0.84118, saving model to /kaggle/working/resnet101_kfold/fold_0/resnet101.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 27s 210ms/step - accuracy: 0.6046 - loss: 1.2116 - val_accuracy: 0.8412 - val_loss: 0.6974
Epoch 4/6

# CareNet

In [1]:
import os
import random
import numpy as np
import tensorflow as tf
import shutil
from tensorflow.keras import layers, callbacks, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import StratifiedKFold, train_test_split
from keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy


# =================== CUSTOM LAYERS ===================
@tf.keras.utils.register_keras_serializable()
class MeanChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=-1, keepdims=True)

@tf.keras.utils.register_keras_serializable()
class MaxChannel(layers.Layer):
    def call(self, inputs):
        return tf.reduce_max(inputs, axis=-1, keepdims=True)

# =================== CONSTANTS ===================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 60
K_FOLDS = 5
SEED = 42

# =================== PATHS ===================
dataset_dir = '/kaggle/input/colon_augmented_png_v9'
output_base_dir = '/kaggle/working/colon_split'
train_val_dir = os.path.join(output_base_dir, 'train_val')
test_dir = os.path.join(output_base_dir, 'test')

# =================== CLEAN EXISTING SPLITS ===================
if os.path.exists(output_base_dir):
    shutil.rmtree(output_base_dir)

# =================== SPLIT DATA INTO TRAIN_VAL AND TEST ===================
os.makedirs(train_val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

class_names = sorted(os.listdir(dataset_dir))

TEST_RATIO = 0.15

print("Splitting dataset into train_val and test sets...")
for class_name in class_names:
    class_input_dir = os.path.join(dataset_dir, class_name)
    images = sorted(os.listdir(class_input_dir))

    train_val_imgs, test_imgs = train_test_split(images, test_size=TEST_RATIO, random_state=SEED, shuffle=True)

    os.makedirs(os.path.join(train_val_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    for img in train_val_imgs:
        shutil.copy(os.path.join(class_input_dir, img), os.path.join(train_val_dir, class_name, img))
    for img in test_imgs:
        shutil.copy(os.path.join(class_input_dir, img), os.path.join(test_dir, class_name, img))

print("Dataset split done.")
print(f"Train_Val directory: {train_val_dir}")
print(f"Test directory: {test_dir}")

# =================== COLLECT ALL TRAIN_VAL IMAGES AND LABELS ===================
image_paths = []
labels = []
for idx, class_name in enumerate(class_names):
    class_dir = os.path.join(train_val_dir, class_name)
    imgs = os.listdir(class_dir)
    for img in imgs:
        image_paths.append(os.path.join(class_dir, img))
        labels.append(idx)

image_paths = np.array(image_paths)
labels = np.array(labels)

# =================== MODEL BUILDING ===================
def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)

    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)

    mlp_avg = shared_dense_two(shared_dense_one(avg_pool))
    mlp_max = shared_dense_two(shared_dense_one(max_pool))

    channel_attention = layers.Add()([mlp_avg, mlp_max])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channel))(channel_attention)

    x = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = MeanChannel()(x)
    max_pool_spatial = MaxChannel()(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)

    refined_feature = layers.Multiply()([x, spatial_attention])
    return refined_feature

def build_model():
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False, input_shape=(224, 224, 3), weights='imagenet'
    )
    base_model.trainable = True

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=True)
    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

# =================== DATA GENERATORS ===================
val_test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =================== STRATIFIED K-FOLD TRAINING ===================
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
test_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels)):
    print(f"\n📁 Fold {fold + 1}/{K_FOLDS}")

    # Create fold directory structure
    fold_dir = os.path.join(output_base_dir, f'fold_{fold}')
    train_dir = os.path.join(fold_dir, 'train')
    val_dir = os.path.join(fold_dir, 'val')

    for dir_path in [train_dir, val_dir]:
        for class_name in class_names:
            os.makedirs(os.path.join(dir_path, class_name), exist_ok=True)

    # Copy train files
    for i in train_idx:
        src = image_paths[i]
        class_name = class_names[labels[i]]
        dst = os.path.join(train_dir, class_name, os.path.basename(src))
        shutil.copy(src, dst)

    # Copy val files
    for i in val_idx:
        src = image_paths[i]
        class_name = class_names[labels[i]]
        dst = os.path.join(val_dir, class_name, os.path.basename(src))
        shutil.copy(src, dst)

    # Data generators
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
    )
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=True
    )

    val_generator = val_test_datagen.flow_from_directory(
        val_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    # Build, compile model
    model = build_model()
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=1.5e-3,
        decay_steps=1000,
        alpha=1e-6
    )
    optimizer = AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
    loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

    checkpoint_path = os.path.join(fold_dir, 'best_model.keras')
    checkpoint_cb = callbacks.ModelCheckpoint(
        checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1
    )
    earlystop_cb = callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1
    )

    model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=val_generator,
        callbacks=[checkpoint_cb, earlystop_cb],
        verbose=1
    )

    # Load best model & evaluate on test set
    model.load_weights(checkpoint_path)
    test_loss, test_acc = model.evaluate(test_generator, verbose=1)
    print(f"✅ Fold {fold + 1} Test Accuracy: {test_acc:.4f}")
    test_accuracies.append(test_acc)

    # Clear resources for next fold
    del model
    K.clear_session()

print("\n📊 Cross-Validation Test Accuracy Summary:")
for i, acc in enumerate(test_accuracies):
    print(f"  Fold {i+1}: {acc:.4f}")

print(f"\n✅ Average Test Accuracy: {np.mean(test_accuracies):.4f}")


2025-07-28 13:46:45.985634: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753710406.172858      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753710406.227104      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Splitting dataset into train_val and test sets...
Dataset split done.
Train_Val directory: /kaggle/working/colon_split/train_val
Test directory: /kaggle/working/colon_split/test
Found 900 images belonging to 4 classes.

📁 Fold 1/5
Found 4080 images belonging to 4 classes.
Found 1020 images belonging to 4 classes.


I0000 00:00:1753710466.885836      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1753710466.886577      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/60


I0000 00:00:1753710538.570640     117 service.cc:148] XLA service 0x7c37d4005340 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753710538.571456     117 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1753710538.571480     117 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1753710545.480918     117 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1753710556.410439     117 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1753710556.555521     117 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1753710557.007098     117 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. Th

 72/128 ━━━━━━━━━━━━━━━━━━━━ 7s 125ms/step - accuracy: 0.7432 - loss: 0.7956

E0000 00:00:1753710602.774167     119 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1753710602.914018     119 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1753710603.251319     119 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1753710603.392711     119 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 414ms/step - accuracy: 0.8017 - loss: 0.7093
Epoch 1: val_accuracy improved from -inf to 0.25000, saving model to /kaggle/working/colon_split/fold_0/best_model.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 181s 554ms/step - accuracy: 0.8025 - loss: 0.7082 - val_accuracy: 0.2500 - val_loss: 2.4374
Epoch 2/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.9650 - loss: 0.4374
Epoch 2: val_accuracy improved from 0.25000 to 0.26078, saving model to /kaggle/working/colon_split/fold_0/best_model.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 21s 159ms/step - accuracy: 0.9650 - loss: 0.4373 - val_accuracy: 0.2608 - val_loss: 1.7765
Epoch 3/60
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.9828 - loss: 0.3943
Epoch 3: val_accuracy improved from 0.26078 to 0.47157, saving model to /kaggle/working/colon_split/fold_0/best_model.keras
128/128 ━━━━━━━━━━━━━━━━━━━━ 20s 157ms/step - accuracy: 0.9828 - loss: 0.3943 - val_accuracy: 0.4716 - val_loss: 1.5177
Epoch 4/60
128/12

In [5]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

# Accuracy values
carenet = np.array([0.9844, 0.9800, 0.9800, 0.9844, 0.9867])
inceptionresnetv2 = np.array([0.9444, 0.9489, 0.9456, 0.9478, 0.9378])
xception = np.array([0.9578, 0.9589, 0.9589, 0.9556, 0.9600])
convnexttiny = np.array([0.7511, 0.7944, 0.9044, 0.8244, 0.8711])
resnet101 = np.array([0.9644, 0.9678, 0.9578, 0.9611, 0.9511])

# Cohen's d function
def cohens_d(a, b):
    n1, n2 = len(a), len(b)
    pooled_std = np.sqrt(((n1 - 1)*np.var(a, ddof=1) + (n2 - 1)*np.var(b, ddof=1)) / (n1 + n2 - 2))
    return (np.mean(a) - np.mean(b)) / pooled_std

# Prepare comparisons dictionary
comparisons = {
    "Carenet vs InceptionResNetV2": inceptionresnetv2,
    "Carenet vs Xception": xception,
    "Carenet vs ConvNeXt-Tiny": convnexttiny,
    "Carenet vs ResNet101": resnet101
}

# Calculate statistics and store results
results = []
for name, values in comparisons.items():
    t_stat, p_val = ttest_ind(carenet, values, equal_var=False)
    d = cohens_d(carenet, values)
    results.append({
        "Compared Models": name,
        "t-statistic": f"{t_stat:.4f}",
        "p-value": f"{p_val:.6f}",
        "Cohen’s d (Effect Size)": f"{d:.4f}"
    })

# Print a professional Markdown table
print("\n### 📊 Model Comparison: Carenet vs Others\n")
print("| **Compared Models**             | **t-statistic** | **p-value** | **Cohen’s d (Effect Size)** |")
print("|--------------------------------|----------------:|------------:|-----------------------------:|")
for row in results:
    print(f"| {row['Compared Models']:<30} | {row['t-statistic']:>14} | {row['p-value']:>10} | {row['Cohen’s d (Effect Size)']:>27} |")



### 📊 Model Comparison: Carenet vs Others

| **Compared Models**             | **t-statistic** | **p-value** | **Cohen’s d (Effect Size)** |
|--------------------------------|----------------:|------------:|-----------------------------:|
| Carenet vs InceptionResNetV2   |        16.2062 |   0.000001 |                     10.2497 |
| Carenet vs Xception            |        16.2703 |   0.000002 |                     10.2902 |
| Carenet vs ConvNeXt-Tiny       |         5.6658 |   0.004716 |                      3.5833 |
| Carenet vs ResNet101           |         7.1648 |   0.000487 |                      4.5314 |
